# Master Data Management (MDM) Showcase

## Snowflake-Native MDM Solution

This notebook demonstrates a complete **Customer MDM** solution built entirely on Snowflake, featuring:

- **Entity Resolution**: Identifying and linking duplicate customers across CRM systems
- **Survivorship Rules**: Selecting the best attribute values from matched records
- **Golden Records**: Creating a single source of truth for each customer
- **SCD Type 2 History**: Tracking all changes over time with valid_from/valid_to
- **Address Consolidation**: ONE golden address per customer (1:1 relationship)

### Architecture
```
RAW Layer (CRM_A, CRM_B) → AGGREGATION Layer (MDM) → SERVICE Layer (Customer 360)
```

In [ ]:
%%sql -r datafsetrame_1
-- Setup: Connect to MDM database and schema
USE DATABASE MDM_DEV;
USE SCHEMA MDM_AGG_001;
USE WAREHOUSE MD_TEST_WH;

---
## 1. Data Volume Overview

Let's examine the data volumes across our MDM pipeline:
- **RAW Layer**: Source records from CRM_A and CRM_B (append-only)
- **Golden Records**: Deduplicated master records after entity resolution
- **History Records**: SCD Type 2 tracking all changes over time

In [ ]:
%%sql -r dataframe_2
-- Data volume summary across MDM layers
SELECT 'RAW: CUSTOMER_A' AS layer, COUNT(*) AS records FROM MDM_RAW_001.CRMI_RAW_TB_CUSTOMER_A
UNION ALL SELECT 'RAW: CUSTOMER_B', COUNT(*) FROM MDM_RAW_001.CRMI_RAW_TB_CUSTOMER_B
UNION ALL SELECT 'RAW: ADDRESSES_A', COUNT(*) FROM MDM_RAW_001.CRMI_RAW_TB_ADDRESSES_A
UNION ALL SELECT 'RAW: ADDRESSES_B', COUNT(*) FROM MDM_RAW_001.CRMI_RAW_TB_ADDRESSES_B
UNION ALL SELECT '─────────────────', NULL
UNION ALL SELECT 'GOLDEN: CUSTOMERS', COUNT(*) FROM CRMA_AGG_DT_CUSTOMER
UNION ALL SELECT 'GOLDEN: ADDRESSES', COUNT(*) FROM CRMA_AGG_DT_ADDRESSES
UNION ALL SELECT '─────────────────', NULL
UNION ALL SELECT 'HISTORY: CUSTOMERS', COUNT(*) FROM CRMA_AGG_DT_CUSTOMER_HISTORY
UNION ALL SELECT 'HISTORY: ADDRESSES', COUNT(*) FROM CRMA_AGG_DT_ADDRESSES_HISTORY;

---
## 2. Entity Resolution: Finding Duplicate Customers

Entity resolution identifies customers that exist in **both** CRM systems. Our matching rules include:

| Rule | Logic | Confidence |
|------|-------|------------|
| **MATCH-D01** | Exact email match | 100% |
| **MATCH-D02** | Phone (last 10 digits) | 95% |
| **MATCH-P01** | Name similarity (Jaro-Winkler ≥85%) | 85% |
| **MATCH-P02** | SOUNDEX last name match | Probabilistic |

Records with combined score ≥70% are merged into a single golden record.

In [ ]:
%%sql -r dataframe_3
-- Entity resolution statistics: How many customers matched across systems?
SELECT 
    source_count,
    CASE source_count
        WHEN 1 THEN 'Single-source customers (no match found)'
        WHEN 2 THEN 'Merged customers (best attributes from BOTH CRM_A and CRM_B)'
        ELSE 'Multi-source (' || source_count || ' sources)'
    END AS description,
    COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS percentage
FROM CRMA_AGG_DT_CUSTOMER
GROUP BY source_count
ORDER BY source_count;

---
## 3. Survivorship Rules: Which Source Wins?

When a customer exists in multiple sources, **survivorship rules** determine which attribute value becomes the golden record:

| Attribute | Survivorship Rule | Rationale |
|-----------|-------------------|----------|
| `first_name` | Non-empty value → Source priority (CRM_A) → Recency | Complete data preferred |
| `last_name` | Non-empty value → Source priority → Recency | Complete data preferred |
| `email` | Valid format → Source priority (CRM_A) → Recency | CRM_A is system of record |
| `phone` | Length ≥7 digits → Source priority → Recency | More digits = more complete |

### Example: Survivorship "Race" - CRM_A vs CRM_B

In [ ]:
%%sql -r dataframe_4
-- Survivorship race: Show customers in BOTH systems with field-by-field winners
WITH matched_customers AS (
    SELECT customer_id
    FROM CRMA_AGG_VW_CUSTOMER_GROUPS
    GROUP BY customer_id
    HAVING COUNT(DISTINCT source_system) = 2
),
crm_a_data AS (
    SELECT g.customer_id, u.first_name, u.last_name, u.email, u.phone
    FROM CRMA_AGG_VW_CUSTOMER_GROUPS g
    JOIN CRMA_AGG_VW_CUSTOMER_UNION u ON g.source_system = u.source_system AND g.source_key = u.source_key
    WHERE g.source_system = 'CRM_A'
),
crm_b_data AS (
    SELECT g.customer_id, u.first_name, u.last_name, u.email, u.phone
    FROM CRMA_AGG_VW_CUSTOMER_GROUPS g
    JOIN CRMA_AGG_VW_CUSTOMER_UNION u ON g.source_system = u.source_system AND g.source_key = u.source_key
    WHERE g.source_system = 'CRM_B'
)
SELECT 
    m.customer_id,
    a.first_name AS crm_a_first, b.first_name AS crm_b_first, g.first_name AS golden_first,
    CASE 
        WHEN a.first_name = b.first_name THEN '✓ MATCH'
        WHEN g.first_name = a.first_name THEN '← CRM_A wins'
        WHEN g.first_name = b.first_name THEN '→ CRM_B wins'
        ELSE '?' 
    END AS first_name_winner,
    a.email AS crm_a_email, b.email AS crm_b_email, g.email AS golden_email,
    CASE 
        WHEN a.email = b.email THEN '✓ MATCH'
        WHEN g.email = a.email THEN '← CRM_A wins'
        WHEN g.email = b.email THEN '→ CRM_B wins'
        ELSE '?' 
    END AS email_winner
FROM matched_customers m
JOIN crm_a_data a ON m.customer_id = a.customer_id
JOIN crm_b_data b ON m.customer_id = b.customer_id
JOIN CRMA_AGG_DT_CUSTOMER g ON m.customer_id = g.customer_id
ORDER BY m.customer_id
LIMIT 15;

---
## 4. Survivorship Winner Summary

Aggregate view: How often does each source "win" for each field across all matched customers?

In [ ]:
%%sql -r dataframe_5
-- Survivorship winner summary: Which source wins most often per field?
WITH matched_customers AS (
    SELECT customer_id FROM CRMA_AGG_VW_CUSTOMER_GROUPS GROUP BY customer_id HAVING COUNT(DISTINCT source_system) = 2
),
crm_a_data AS (
    SELECT g.customer_id, u.first_name, u.last_name, u.email, u.phone
    FROM CRMA_AGG_VW_CUSTOMER_GROUPS g
    JOIN CRMA_AGG_VW_CUSTOMER_UNION u ON g.source_system = u.source_system AND g.source_key = u.source_key
    WHERE g.source_system = 'CRM_A'
),
crm_b_data AS (
    SELECT g.customer_id, u.first_name, u.last_name, u.email, u.phone
    FROM CRMA_AGG_VW_CUSTOMER_GROUPS g
    JOIN CRMA_AGG_VW_CUSTOMER_UNION u ON g.source_system = u.source_system AND g.source_key = u.source_key
    WHERE g.source_system = 'CRM_B'
),
comparisons AS (
    SELECT 
        CASE WHEN gc.first_name = a.first_name THEN 'CRM_A' WHEN gc.first_name = b.first_name THEN 'CRM_B' ELSE 'OTHER' END AS first_name_winner,
        CASE WHEN gc.last_name = a.last_name THEN 'CRM_A' WHEN gc.last_name = b.last_name THEN 'CRM_B' ELSE 'OTHER' END AS last_name_winner,
        CASE WHEN gc.email = a.email THEN 'CRM_A' WHEN gc.email = b.email THEN 'CRM_B' ELSE 'OTHER' END AS email_winner,
        CASE WHEN gc.phone = a.phone THEN 'CRM_A' WHEN gc.phone = b.phone THEN 'CRM_B' ELSE 'OTHER' END AS phone_winner
    FROM matched_customers m
    JOIN crm_a_data a ON m.customer_id = a.customer_id
    JOIN crm_b_data b ON m.customer_id = b.customer_id
    JOIN CRMA_AGG_DT_CUSTOMER gc ON m.customer_id = gc.customer_id
)
SELECT 
    'FIRST_NAME' AS field, SUM(IFF(first_name_winner='CRM_A',1,0)) AS crm_a_wins, SUM(IFF(first_name_winner='CRM_B',1,0)) AS crm_b_wins, COUNT(*) AS total
FROM comparisons
UNION ALL SELECT 'LAST_NAME', SUM(IFF(last_name_winner='CRM_A',1,0)), SUM(IFF(last_name_winner='CRM_B',1,0)), COUNT(*) FROM comparisons
UNION ALL SELECT 'EMAIL', SUM(IFF(email_winner='CRM_A',1,0)), SUM(IFF(email_winner='CRM_B',1,0)), COUNT(*) FROM comparisons
UNION ALL SELECT 'PHONE', SUM(IFF(phone_winner='CRM_A',1,0)), SUM(IFF(phone_winner='CRM_B',1,0)), COUNT(*) FROM comparisons;

---
## 5. Golden Customer Records

The **Golden Record** is the single source of truth for each customer, containing the best surviving values from all matched source records.

Key columns:
- `customer_id`: Unique master identifier
- `source_count`: Number of source systems (1=single, 2=matched)
- `dq_score`: Data quality score (0-100)
- `last_updated`: Most recent change timestamp

In [ ]:
%%sql -r dataframe_6
-- Sample golden customer records
SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    dq_score,
    source_count,
    last_updated
FROM CRMA_AGG_DT_CUSTOMER
WHERE source_count = 2
ORDER BY customer_id
LIMIT 15;

---
## 6. Data Quality Distribution

The DQ Score reflects data completeness and validity:
- **100**: Complete record (valid email + phone + first_name + last_name)
- **75**: Has email and phone
- **50**: Has email or phone
- **25**: Missing contact information

In [ ]:
%%sql -r dataframe_7
-- Data quality score distribution
SELECT 
    dq_score,
    CASE dq_score
        WHEN 100 THEN 'Excellent - Complete record'
        WHEN 75 THEN 'Good - Has email and phone'
        WHEN 50 THEN 'Fair - Has email or phone'
        ELSE 'Poor - Missing contact info'
    END AS quality_tier,
    COUNT(*) AS customer_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS percentage
FROM CRMA_AGG_DT_CUSTOMER
GROUP BY dq_score
ORDER BY dq_score DESC;

---
## 7. SCD Type 2 History: Tracking Changes Over Time

The history table tracks **all changes** to customer records using Slowly Changing Dimension Type 2:
- `valid_from`: Date this version became active
- `valid_to`: Date this version was superseded (9999-12-31 = current record)

### Customers with Multiple Historical Versions

In [ ]:
%%sql -r dataframe_8
-- Customers with the most historical versions (most changes over time)
SELECT 
    customer_id, 
    COUNT(*) AS version_count,
    MIN(valid_from) AS first_seen,
    MAX(valid_from) AS last_changed
FROM CRMA_AGG_DT_CUSTOMER_HISTORY
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY version_count DESC
LIMIT 10;

### Example: Complete History for a Single Customer

See how customer attributes changed over time:

In [ ]:
%%sql -r dataframe_9
-- Complete history for customer with multiple versions
WITH top_customer AS (
    SELECT customer_id FROM CRMA_AGG_DT_CUSTOMER_HISTORY GROUP BY customer_id ORDER BY COUNT(*) DESC LIMIT 1
)
SELECT 
    h.customer_id,
    h.first_name,
    h.last_name,
    h.email,
    h.phone,
    h.dq_score,
    h.valid_from,
    h.valid_to,
    CASE WHEN h.valid_to = '9999-12-31' THEN '← CURRENT' ELSE '' END AS status
FROM CRMA_AGG_DT_CUSTOMER_HISTORY h
JOIN top_customer t ON h.customer_id = t.customer_id
ORDER BY h.valid_from;

---
## 8. Address Consolidation: ONE Address Per Customer

Our MDM solution creates **exactly ONE golden address per customer** by merging addresses from all sources.

Key design decisions:
- `address_id = customer_id` (1:1 relationship)
- Field-level survivorship picks the best value from any source
- `is_primary = TRUE` for all (only one address per customer)

In [ ]:
%%sql -r dataframe_10
-- Golden addresses with customer linkage
SELECT 
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    c.email,
    a.address_id,
    a.street,
    a.city,
    a.postal_code,
    a.country,
    a.is_primary,
    CASE WHEN a.address_id = c.customer_id THEN '✓ 1:1 valid' ELSE '✗ ERROR' END AS relationship_check
FROM CRMA_AGG_DT_CUSTOMER c
JOIN CRMA_AGG_DT_ADDRESSES a ON c.customer_id = a.customer_id
WHERE c.source_count = 2
ORDER BY c.customer_id
LIMIT 15;

### Address Survivorship Summary

For matched customers with addresses in both systems, which source wins each field?

In [ ]:
%%sql -r dataframe_11
-- Address survivorship summary
WITH matched_customers AS (
    SELECT customer_id FROM CRMA_AGG_VW_CUSTOMER_GROUPS GROUP BY customer_id HAVING COUNT(DISTINCT source_system) = 2
),
crm_a_addr AS (
    SELECT cg.customer_id, au.street, au.city, au.postal_code, au.country
    FROM matched_customers mc
    JOIN CRMA_AGG_VW_CUSTOMER_GROUPS cg ON mc.customer_id = cg.customer_id AND cg.source_system = 'CRM_A'
    JOIN CRMA_AGG_VW_ADDRESSES_UNION au ON au.source_system = 'CRM_A' AND au.source_customer_key = cg.source_key
),
crm_b_addr AS (
    SELECT cg.customer_id, au.street, au.city, au.postal_code, au.country
    FROM matched_customers mc
    JOIN CRMA_AGG_VW_CUSTOMER_GROUPS cg ON mc.customer_id = cg.customer_id AND cg.source_system = 'CRM_B'
    JOIN CRMA_AGG_VW_ADDRESSES_UNION au ON au.source_system = 'CRM_B' AND au.source_customer_key = cg.source_key
),
comparisons AS (
    SELECT 
        CASE WHEN g.street = a.street THEN 'CRM_A' WHEN g.street = b.street THEN 'CRM_B' ELSE 'OTHER' END AS street_winner,
        CASE WHEN g.city = a.city THEN 'CRM_A' WHEN g.city = b.city THEN 'CRM_B' ELSE 'OTHER' END AS city_winner,
        CASE WHEN g.country = a.country THEN 'CRM_A' WHEN g.country = b.country THEN 'CRM_B' ELSE 'OTHER' END AS country_winner
    FROM crm_a_addr a
    JOIN crm_b_addr b ON a.customer_id = b.customer_id
    JOIN CRMA_AGG_DT_ADDRESSES g ON a.customer_id = g.customer_id
)
SELECT 'STREET' AS field, SUM(IFF(street_winner='CRM_A',1,0)) AS crm_a_wins, SUM(IFF(street_winner='CRM_B',1,0)) AS crm_b_wins FROM comparisons
UNION ALL SELECT 'CITY', SUM(IFF(city_winner='CRM_A',1,0)), SUM(IFF(city_winner='CRM_B',1,0)) FROM comparisons
UNION ALL SELECT 'COUNTRY', SUM(IFF(country_winner='CRM_A',1,0)), SUM(IFF(country_winner='CRM_B',1,0)) FROM comparisons;

---
## 9. Customer 360 View: Complete Picture

The Customer 360 combines golden customer records with their golden addresses for a complete view.

In [ ]:
%%sql -r dataframe_12
-- Customer 360 View: Golden customer + Golden address
SELECT 
    c.customer_id,
    c.first_name,
    c.last_name,
    c.email,
    c.phone,
    c.dq_score AS customer_dq,
    c.source_count,
    '│' AS "|",
    a.street,
    a.city,
    a.postal_code,
    a.country
FROM CRMA_AGG_DT_CUSTOMER c
LEFT JOIN CRMA_AGG_DT_ADDRESSES a ON c.customer_id = a.customer_id
ORDER BY c.customer_id
LIMIT 20;

---
## 10. Point-in-Time Query: Historical State

Using SCD Type 2, we can query the state of any customer at any point in time.

In [ ]:
%%sql -r dataframe_13
-- Point-in-time query: Customer state on a specific date
-- Change the date to explore different historical states
SET query_date = '2026-02-10';

SELECT 
    customer_id,
    first_name,
    last_name,
    email,
    phone,
    dq_score,
    valid_from,
    valid_to
FROM CRMA_AGG_DT_CUSTOMER_HISTORY
WHERE $query_date BETWEEN valid_from AND valid_to
ORDER BY customer_id
LIMIT 20;

---
## 11. Source-to-Golden Lineage

Full traceability showing how records from both source systems map to golden records:

In [ ]:
%%sql -r dataframe_14
-- Source-to-Golden lineage: Track how records merge
SELECT 
    g.customer_id AS golden_customer_id,
    gc.first_name || ' ' || gc.last_name AS golden_name,
    gc.email AS golden_email,
    gc.source_count,
    '←' AS from_source_system,
    g.source_system,
    g.source_key,
    u.first_name AS source_first_name,
    u.email AS source_email
FROM CRMA_AGG_VW_CUSTOMER_GROUPS g
JOIN CRMA_AGG_DT_CUSTOMER gc ON g.customer_id = gc.customer_id
JOIN CRMA_AGG_VW_CUSTOMER_UNION u ON g.source_system = u.source_system AND g.source_key = u.source_key
WHERE gc.source_count = 2
ORDER BY g.customer_id, g.source_system
LIMIT 30;